# Relation labels: baselines and evaluation

Every method is scored against the Opus labels of the 1,000 test instances. The baselines are developed on `dev` (10% of the Qwen labels) and see `test` once. PubMedBERT is trained in `training.ipynb`.

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, precision_recall_fscore_support
from sklearn.pipeline import make_pipeline
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from textmining.bert import MAX_LEN, allowed_masks, masked_logits, tokenize
from textmining.relation_data import ALLOWED_LABELS, LABELS, load_all, split_dev

RUNS = '/mnt/raidbio2/extstud/studtemp/mitsopoulos/mirna-annotation/runs'
MODEL_DIR = '../data/models/relation_pubmedbert'
FIGURES = '/mnt/raidbio2/extstud/studtemp/mitsopoulos/bachelor-thesis/figures'
TOKENS = r'(?u)\$?\b\w\w+\b'      # words of two or more characters; keeps the $ of $GENE etc.


def scores(y, p, labels=LABELS):
    y, p = np.asarray(y), np.asarray(p)
    return np.mean(y == p), cohen_kappa_score(y, p), f1_score(y, p, labels=labels, average='macro', zero_division=0)


def interval(y, p, labels=LABELS, resamples=1000, seed=42):
    # 95% percentile bootstrap of scores()
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    draws = [scores(y[i], p[i], labels) for i in (rng.integers(0, len(y), len(y)) for _ in range(resamples))]
    return np.percentile(draws, [2.5, 97.5], axis=0)


def allowed_only(proba, classes, entity_types):
    # the most probable label that is allowed for the entity type
    allowed = np.array([[c in ALLOWED_LABELS[t] for c in classes] for t in entity_types])
    return np.asarray(classes)[np.where(allowed, proba, -1).argmax(1)]

## Data

In [ ]:
train, test = load_all()
fit, dev = split_dev(train)
test['clean'] = ~test['error'].fillna('').str.contains('unclear|ner_error')
len(train), len(fit), len(dev), len(test), test['clean'].sum()

## Majority label per entity type

In [ ]:
majority = train.groupby('entity_type')['label'].agg(lambda s: s.mode()[0])
test['majority'] = test['entity_type'].map(majority)

## Trigger-word rules

Candidate trigger words per label, ranked by lift on `fit`: the share of a word's occurrences that fall in the label, divided by the label's share of all words. The lexicon `trigger_words.syn` was written from these candidates.

In [ ]:
vectorizer = CountVectorizer(token_pattern=TOKENS, ngram_range=(1, 2))
X = vectorizer.fit_transform(fit['sentence'])
counts = pd.DataFrame({label: X[(fit['label'] == label).values].sum(axis=0).A1 for label in LABELS},
                      index=vectorizer.get_feature_names_out()).T
share = counts / counts.sum(axis=0)
lift = share.div(counts.sum(axis=1) / counts.values.sum(), axis=0)


def candidates(label, min_count=20, n=50):
    table = pd.DataFrame({'count': counts.loc[label], 'share': share.loc[label], 'lift': lift.loc[label]})
    return table[table['count'] >= min_count].nlargest(n, 'lift').round(2)


pd.DataFrame({label: pd.Series(candidates(label, n=15).index) for label in LABELS})

In [ ]:
lexicon = {}
for line in open('trigger_words.syn'):
    if line.strip():
        label, words = line.strip().split(':', 1)
        lexicon.setdefault(label, []).extend(w for w in words.split('|') if w)
patterns = {label: re.compile(r'\b(?:' + '|'.join(map(re.escape, words)) + r')\b', re.IGNORECASE)
            for label, words in lexicon.items()}
PRIORITY = ['biomarker', 'targets', 'promotes', 'inhibits', 'expression_up', 'expression_down', 'expressed', 'associated']


def rules(data):
    # first label in PRIORITY that is allowed for the entity type and has a trigger in the sentence
    return [next((label for label in PRIORITY if label in ALLOWED_LABELS[t] and patterns[label].search(s)), 'none')
            for s, t in zip(data['sentence'], data['entity_type'])]


test['rules'] = rules(test)
scores(dev['label'], rules(dev))

In [ ]:
# lexicon entries per label, and the mean lift of their distinct uni- and bigrams on fit
analyze = vectorizer.build_analyzer()
entries = pd.DataFrame([(label, ' '.join(t for t in analyze(w) if ' ' not in t)) for label, words in lexicon.items() for w in words],
                       columns=['label', 'ngram'])
found = entries[entries['ngram'].isin(lift.columns)].drop_duplicates()
found['lift'] = [lift.at[label, ngram] for label, ngram in zip(found['label'], found['ngram'])]
pd.DataFrame({'entries': entries.groupby('label').size(), 'mean_lift': found.groupby('label')['lift'].mean(),
              'max_lift': counts.values.sum() / counts.sum(axis=1)}).loc[PRIORITY].astype({'entries': int}).round(2)

## Word n-gram model

Logistic regression on TF-IDF weighted word uni- and bigrams of the entity type and the masked sentence. C is chosen on `dev` by macro-F1, then the model is refitted on all Qwen labels.

In [ ]:
def text(data):
    return data['entity_type'] + ' ' + data['sentence']


def ngram_model(C):
    return make_pipeline(TfidfVectorizer(token_pattern=TOKENS, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(C=C, max_iter=3000))


def ngram_predict(model, data):
    return allowed_only(model.predict_proba(text(data)), model.classes_, data['entity_type'])


dev_scores = {C: scores(dev['label'], ngram_predict(ngram_model(C).fit(text(fit), fit['label']), dev)) for C in (1, 4, 16)}
best_C = max(dev_scores, key=lambda C: dev_scores[C][2])
ngram = ngram_model(best_C).fit(text(train), train['label'])
test['ngram'] = ngram_predict(ngram, test)
pd.DataFrame(dev_scores, index=['agreement', 'kappa', 'macro_f1']).round(3)

## PubMedBERT (saved model from `training.ipynb`)

In [ ]:
assert json.load(open(f'{MODEL_DIR}/settings.json'))['mask_entities'] == 'all'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
bert = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
assert [bert.config.id2label[i] for i in range(len(LABELS))] == LABELS
allowed = allowed_masks(ALLOWED_LABELS, LABELS)


def bert_predict(data, batch_size=64):
    predicted = []
    with torch.no_grad():
        for start in range(0, len(data), batch_size):
            rows = data.iloc[start:start + batch_size]
            batch = tokenizer.pad([tokenize(tokenizer, t, s, MAX_LEN) for t, s in zip(rows['entity_type'], rows['sentence'])],
                                  return_tensors='pt')
            mask = torch.stack([allowed[t] for t in rows['entity_type']])
            predicted += masked_logits(bert, batch, mask).argmax(-1).tolist()
    return [LABELS[i] for i in predicted]


test['bert'] = bert_predict(test)

## LLM annotators (invalid outputs count as disagreements)

In [ ]:
def llm_labels(path):
    rows = (json.loads(line) for line in open(path))
    return {r['sentence_id']: r['label'] if r.get('valid', True) else 'invalid' for r in rows}


test['qwen'] = test['sentence_id'].map(llm_labels(f'{RUNS}/qwen35_9b_v5_balanced/full/qwen_out.jsonl'))
test['haiku'] = test['sentence_id'].map(llm_labels(f'{RUNS}/haiku55_balanced_v06/model_out.jsonl'))
assert test[['qwen', 'haiku']].notna().all().all()

## Comparison

In [ ]:
METHODS = ['majority', 'rules', 'ngram', 'bert', 'qwen', 'haiku']
rows = []
for subset, part in (('all', test), ('clean', test[test['clean']])):
    pairs = [(m, part['label'], part[m]) for m in METHODS] + [('bert vs qwen', part['qwen'], part['bert'])]
    for method, y, p in pairs:
        point, (low, high) = scores(y, p), interval(y, p)
        rows.append({'subset': subset, 'method': method, 'n': len(y)}
                    | {m: f'{v:.3f} [{lo:.3f}, {hi:.3f}]' for m, v, lo, hi in zip(('agreement', 'kappa', 'macro_f1'), point, low, high)})
pd.DataFrame(rows).set_index(['subset', 'method'])

In [ ]:
# macro-F1 per entity type, over the labels allowed for the type
rows = []
for entity_type, part in test.groupby('entity_type'):
    labels = ALLOWED_LABELS[entity_type]
    for method in ('qwen', 'bert', 'ngram', 'rules'):
        low, high = interval(part['label'], part[method], labels)[:, 2]
        rows.append({'entity_type': entity_type, 'n': len(part), 'method': method,
                     'macro_f1': f"{scores(part['label'], part[method], labels)[2]:.2f} [{low:.2f}, {high:.2f}]"})
pd.DataFrame(rows).pivot(index=['entity_type', 'n'], columns='method', values='macro_f1')

In [ ]:
# F1 per label
f1 = pd.DataFrame({m: f1_score(test['label'], test[m], labels=LABELS, average=None, zero_division=0)
                   for m in ('qwen', 'bert', 'ngram', 'rules')}, index=LABELS)
f1.assign(n=test['label'].value_counts()).sort_values('n', ascending=False).round(2)

In [ ]:
# student vs teacher: which instances each gets right
for subset, part in (('all', test), ('clean', test[test['clean']])):
    qwen_right, bert_right = part['qwen'] == part['label'], part['bert'] == part['label']
    both_wrong = part[~qwen_right & ~bert_right]
    print(subset, '- both wrong:', len(both_wrong), 'with the same label:', (both_wrong['qwen'] == both_wrong['bert']).sum())
    display(pd.crosstab(qwen_right.rename('qwen right'), bert_right.rename('bert right'), margins=True))

## Thesis figures

In [ ]:
plt.rcParams.update({'font.size': 9, 'pdf.fonttype': 42, 'axes.spines.top': False, 'axes.spines.right': False})
NAMES = {'qwen': 'Qwen (teacher)', 'bert': 'PubMedBERT (student)', 'rules': 'Trigger words', 'ngram': 'Logistic regression'}
SHORT = ['none', 'assoc.', 'expr. up', 'expr. down', 'expressed', 'biomarker', 'targets', 'promotes', 'inhibits']
ORDER = ['none', 'associated', 'expression_up', 'expression_down', 'expressed', 'biomarker', 'targets', 'promotes', 'inhibits']

fig, axes = plt.subplots(2, 2, figsize=(6.3, 5.8), layout='constrained', sharex=True, sharey=True)
for ax, method in zip(axes.flat, ('qwen', 'bert', 'rules', 'ngram')):
    counts = pd.crosstab(test['label'], test[method]).reindex(index=ORDER, columns=ORDER, fill_value=0)
    share = counts.to_numpy() / test['label'].value_counts().reindex(ORDER).to_numpy()[:, None]
    image = ax.imshow(share, cmap='Blues', vmin=0, vmax=1)
    for i, j in zip(*np.nonzero(counts.to_numpy())):
        ax.text(j, i, counts.iat[i, j], ha='center', va='center', fontsize=6, color='white' if share[i, j] > 0.55 else '0.15')
    ax.set_title(NAMES[method], fontsize=9)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
for ax in axes[1]:
    ax.set_xticks(range(len(ORDER)), SHORT, rotation=60, ha='right', fontsize=7)
    ax.set_xlabel('Predicted label', fontsize=8)
for ax in axes[:, 0]:
    ax.set_yticks(range(len(ORDER)), SHORT, fontsize=7)
    ax.set_ylabel('Opus label', fontsize=8)
fig.colorbar(image, ax=axes, shrink=0.5, label='Share of the Opus label')
fig.savefig(f'{FIGURES}/label_confusion.pdf')


In [ ]:
support = test['label'].value_counts().reindex(LABELS, fill_value=0)
order = support.sort_values(kind='stable').index.tolist()          # largest support on top
fig, axes = plt.subplots(1, 3, figsize=(6.3, 3.4), layout='constrained', sharey=True)
y = np.arange(len(order))
SERIES = {'rules': ('Trigger words', '#999999'), 'ngram': ('Logistic regression', '#E69F00'), 'bert': ('PubMedBERT', '#0072B2')}
values = {m: precision_recall_fscore_support(test['label'], test[m], labels=order, zero_division=0)[:3] for m in SERIES}
for k, ax in enumerate(axes):
    row = np.array([values[m][k] for m in SERIES])
    ax.hlines(y, row.min(axis=0), row.max(axis=0), color='0.8', lw=1, zorder=1)   # one line per label
    for m, (name, colour) in SERIES.items():
        ax.plot(values[m][k], y, 'o', ms=5, color=colour, mec='white', mew=0.6, zorder=2, label=name)
axes[0].set_yticks(y, [f"{label.replace('_', ' ')} ({support[label]})" for label in order])
for ax, title in zip(axes, ('Precision', 'Recall', 'F1')):
    ax.set_xlim(0, 1)
    ax.set_title(title, fontsize=9)
    ax.grid(axis='x', color='0.9', lw=0.6)
    ax.set_axisbelow(True)
fig.supxlabel('Against the Opus labels', fontsize=8)
fig.legend(*axes[0].get_legend_handles_labels(), frameon=False, loc='outside upper center', ncol=3, fontsize=8)
fig.savefig(f'{FIGURES}/label_rules.pdf')